In [0]:
from pyspark.sql import functions as F
orders = spark.table("debiprasad_olist_bronze_olist_orders_dataset")
items = spark.table("debiprasad_olist_bronze_olist_order_items_dataset")
payments = spark.table("debiprasad_olist_bronze_olist_order_payments_dataset")
reviews = spark.table("debiprasad_olist_bronze_olist_order_reviews_dataset")
customers = spark.table("debiprasad_olist_bronze_olist_customers_dataset")
sellers = spark.table("debiprasad_olist_bronze_olist_sellers_dataset")
products = spark.table("debiprasad_olist_bronze_olist_products_dataset")
translation = spark.table("debiprasad_olist_bronze_product_category_name_translation")

In [0]:
orders_clean = (
    orders
    .dropDuplicates(["order_id"])
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
)

items_clean = (
    items
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    )
    .dropDuplicates(["order_id", "order_item_id"])
)

payments_clean = (
    payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("payment_value"),
        F.sum("payment_installments").alias("payment_installments")
    )
)

reviews_clean = (
    reviews
    .select(
        "order_id",
        F.expr("try_cast(review_score as double)").alias("review_score")
    )
    .groupBy("order_id")
    .agg(F.avg("review_score").alias("review_score"))
)

customers_clean = customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
).dropDuplicates(["customer_id"])

sellers_clean = sellers.select(
    "seller_id",
    "seller_city",
    "seller_state"
).dropDuplicates(["seller_id"])

products_clean = products.select(
    "product_id",
    "product_category_name"
).dropDuplicates(["product_id"])

translation_clean = translation.select(
    "product_category_name",
    "product_category_name_english"
).dropDuplicates(["product_category_name"])

print("Silver preparation complete")

Silver preparation complete


In [0]:
product_data = (
    products_clean.alias("p")
    .join(
        translation_clean.alias("t"),
        F.col("p.product_category_name") == F.col("t.product_category_name"),
        "left"
    )
    .select(
        F.col("p.product_id").alias("product_id"),
        F.col("p.product_category_name").alias("product_category_name"),
        F.col("t.product_category_name_english").alias("product_category_name_english")
    )
)

silver = (
    orders_clean.alias("o")
    .join(items_clean.alias("i"), "order_id", "left")
    .join(payments_clean.alias("pay"), "order_id", "left")
    .join(reviews_clean.alias("r"), "order_id", "left")
    .join(customers_clean.alias("c"), "customer_id", "left")
    .join(sellers_clean.alias("s"), F.col("i.seller_id") == F.col("s.seller_id"), "left")
    .join(product_data.alias("p"), F.col("i.product_id") == F.col("p.product_id"), "left")
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("c.customer_unique_id"),
        F.col("i.order_item_id"),
        F.col("i.product_id"),
        F.col("i.seller_id"),
        F.col("p.product_category_name"),
        F.col("p.product_category_name_english"),
        F.col("i.price"),
        F.col("i.freight_value"),
        F.col("pay.payment_value"),
        F.col("r.review_score"),
        F.col("o.order_purchase_timestamp"),
        F.col("o.order_delivered_customer_date"),
        F.col("o.order_estimated_delivery_date"),
        F.datediff(
            F.to_date("o.order_delivered_customer_date"),
            F.to_date("o.order_purchase_timestamp")
        ).alias("delivery_days"),
        F.datediff(
            F.to_date("o.order_estimated_delivery_date"),
            F.to_date("o.order_purchase_timestamp")
        ).alias("estimated_delivery_days"),
        F.when(
            F.col("o.order_delivered_customer_date") >
            F.col("o.order_estimated_delivery_date"),
            1
        ).otherwise(0).alias("is_late"),
        F.concat(
            F.year("o.order_purchase_timestamp"),
            F.lit("-Q"),
            F.quarter("o.order_purchase_timestamp")
        ).alias("order_quarter"),
        (
            F.coalesce(F.col("i.price"), F.lit(0.0)) +
            F.coalesce(F.col("i.freight_value"), F.lit(0.0))
        ).alias("revenue")
    )
)

print("Silver dataset created")

Silver dataset created


In [0]:
silver.select(
    "order_id",
    "customer_unique_id",
    "seller_id",
    "product_category_name_english",
    "revenue",
    "delivery_days",
    "is_late",
    "review_score",
    "order_quarter"
).show(10, truncate=False)

+--------------------------------+--------------------------------+--------------------------------+-------------------------------+-------+-------------+-------+------------+-------------+
|order_id                        |customer_unique_id              |seller_id                       |product_category_name_english  |revenue|delivery_days|is_late|review_score|order_quarter|
+--------------------------------+--------------------------------+--------------------------------+-------------------------------+-------+-------------+-------+------------+-------------+
|5ff96c15d0b717ac6ad1f3d77225a350|e2dfa3127fedbbca9707b36304996dab|1900267e848ceeba8fa32d80c1a5f5a8|bed_bath_table                 |32.7   |5            |0      |5.0         |2018-Q3      |
|22d17396ac4c177e0ea83929cd4f845d|ded340acb51cad2b368c3e986d006ba8|ad781527c93d00d89a11eecd9dcad7c1|baby                           |217.78 |16           |0      |5.0         |2017-Q3      |
|0a4a2fccb27bd83a892fa503987a595b|c7a9a76a4b24a7e7

In [0]:
silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("debiprasad_olist_silver_orders")

In [0]:
silver_count = spark.table("debiprasad_olist_silver_orders").count()
print("Silver rows:", silver_count)

Silver rows: 113425
